In [19]:
import duckdb
import pandas as pd

con = duckdb.connect("db/transactions.duckdb")

In [20]:
# DATA OVERVIEW
# figure out what the schema looks like
con.execute("DESCRIBE transactions").df()

,column_name,column_type,null,key,default,extra
0,unnamed:_0,BIGINT,YES,None,None,None
1,trans_date_trans_time,TIMESTAMP_NS,YES,None,None,None
2,cc_num,BIGINT,YES,None,None,None
3,merchant,VARCHAR,YES,None,None,None
4,category,VARCHAR,YES,None,None,None
5,amt,DOUBLE,YES,None,None,None
6,first,VARCHAR,YES,None,None,None
7,last,VARCHAR,YES,None,None,None
8,gender,VARCHAR,YES,None,None,None
9,street,VARCHAR,YES,None,None,None


In [21]:
# get sample of 10 rows
con.execute("SELECT * FROM transactions LIMIT 10").df()

,unnamed:_0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,trans_date,trans_hour,trans_month,trans_day_of_week
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,2019-01-01,0,2019-01,Tuesday
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,2019-01-01,0,2019-01,Tuesday
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,2019-01-01,0,2019-01,Tuesday
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,2019-01-01,0,2019-01,Tuesday
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,2019-01-01,0,2019-01,Tuesday
5,5,2019-01-01 00:04:08,4767265376804500,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,...,1961-06-19,189a841a0a8ba03058526bcfe566aab5,1325376248,40.653382,-76.152667,0,2019-01-01,0,2019-01,Tuesday
6,6,2019-01-01 00:04:42,30074693890476,fraud_Rowe-Vandervort,grocery_net,44.54,Kelsey,Richards,F,889 Sarah Station Suite 624,...,1993-08-16,83ec1cc84142af6e2acf10c44949e720,1325376282,37.162705,-100.153370,0,2019-01-01,0,2019-01,Tuesday
7,7,2019-01-01 00:05:08,6011360759745864,fraud_Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,...,1947-08-21,6d294ed2cc447d2c71c7171a3d54967c,1325376308,38.948089,-78.540296,0,2019-01-01,0,2019-01,Tuesday
8,8,2019-01-01 00:05:18,4922710831011201,fraud_Herzog Ltd,misc_pos,4.27,Heather,Chase,F,6888 Hicks Stream Suite 954,...,1941-03-07,fc28024ce480f8ef21a32d64c93a29f5,1325376318,40.351813,-79.958146,0,2019-01-01,0,2019-01,Tuesday
9,9,2019-01-01 00:06:01,2720830304681674,"fraud_Schoen, Kuphal and Nitzsche",grocery_pos,198.39,Melissa,Aguilar,F,21326 Taylor Squares Suite 708,...,1974-03-28,3b9014ea8fb80bd65de0b1463b00b00e,1325376361,37.179198,-87.485381,0,2019-01-01,0,2019-01,Tuesday


In [22]:
# get data time range
con.execute("SELECT MIN(trans_date), MAX(trans_date) FROM transactions").df()

,min(trans_date),max(trans_date)
0,2019-01-01,2020-06-21


In [23]:
# search for any missing vals
con.execute("""
    SELECT 
        COUNT(*) - COUNT(amt) AS missing_amt,
        COUNT(*) - COUNT(merchant) AS missing_merchant,
        COUNT(*) - COUNT(category) AS missing_category
    FROM transactions
""").df()

,missing_amt,missing_merchant,missing_category
0,0,0,0


In [24]:
# how much do the transaction amounts vary?
con.execute("""
    SELECT 
        MIN(amt), 
        ROUND(AVG(amt), 2) AS avg,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY amt), 2) AS median,
        MAX(amt)
    FROM transactions
""").df()

,min(amt),avg,median,max(amt)
0,1.0,70.35,47.52,28948.9


In [25]:
# how much fraud is occurring compared to normal transactions?
con.execute("""
    SELECT is_fraud, COUNT(*) as count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM transactions
    GROUP BY is_fraud
""").df()

,is_fraud,count,pct
0,0,1289169,99.42
1,1,7506,0.58


In [32]:

con.execute("""
    SELECT 
        is_fraud,
        ROUND(AVG(amt), 2) as avg_amt,
        ROUND(MIN(amt), 2) as min_amt,
        ROUND(MAX(amt), 2) as max_amt,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY amt), 2) as median_amt
    FROM transactions
    GROUP BY is_fraud
""").df()

,is_fraud,avg_amt,min_amt,max_amt,median_amt
0,0,67.67,1.00,28948.90,47.28
1,1,531.32,1.06,1376.04,396.51


In [26]:
# QUESTION 1: Where is fraud occurring? 
# HYPOTHESIS: Fraud is likely concentrated in specific categories where there is less security or other factors make it more beneficial
# over others. Online transactions could be a target since stolen card information can be used without the physical chip being read. 

# 1A: What merchants/categories are fraud most common in? What is the fraud RATE across different categories?
con.execute("""
    SELECT 
        category,
        COUNT(*) as total_transactions,
        SUM(is_fraud) as fraud_count,
        ROUND(100.0 * SUM(is_fraud) / COUNT(*), 2) as fraud_rate_pct
    FROM transactions
    GROUP BY category
    ORDER BY fraud_rate_pct DESC
""").df()

,category,total_transactions,fraud_count,fraud_rate_pct
0,shopping_net,97543,1713.0,1.76
1,misc_net,63287,915.0,1.45
2,grocery_pos,123638,1743.0,1.41
3,shopping_pos,116672,843.0,0.72
4,gas_transport,131659,618.0,0.47
5,misc_pos,79655,250.0,0.31
6,travel,40507,116.0,0.29
7,grocery_net,45452,134.0,0.29
8,entertainment,94014,233.0,0.25
9,personal_care,90758,220.0,0.24


In [ ]:
# FINDINGS: The fraud rate (fraud_rate_pct) is highest in the 'shopping_net' and 'misc_net' categories while 'food_dining', 'home', 'health_fitness'
# have the lowest rate. Those are both online transaction based categories, which support the hypothesis. 

In [36]:
# 1B: Where is fraud happening in highest cost volume? 
con.execute("""
    SELECT 
        category,
        ROUND(SUM(CASE WHEN is_fraud = 1 THEN amt ELSE 0 END), 2) as fraud_dollar_volume,
        ROUND(AVG(CASE WHEN is_fraud = 1 THEN amt ELSE 0 END), 2) as avg_fraud_amt,

    FROM transactions
    GROUP BY category
    ORDER BY fraud_dollar_volume DESC
""").df()

,category,fraud_dollar_volume,avg_fraud_amt
0,shopping_net,1711723.71,17.55
1,shopping_pos,739245.09,6.34
2,misc_net,729266.76,11.52
3,grocery_pos,543797.90,4.40
4,entertainment,117323.79,1.25
5,misc_pos,54571.02,0.69
6,home,50971.66,0.41
7,food_dining,18131.62,0.20
8,gas_transport,7594.11,0.06
9,personal_care,5757.52,0.06


In [ ]:
# FINDINGS: 'shopping_net' and 'misc_net' have the highest fraud cost volume.

In [28]:
# 1C: Is fraud happening across the category or only specific merchants?
con.execute("""
    SELECT 
        merchant,
        COUNT(*) as total_transactions,
        SUM(is_fraud) as fraud_count,
        ROUND(100.0 * SUM(is_fraud) / COUNT(*), 2) as fraud_rate_pct,
        ROUND(SUM(CASE WHEN is_fraud = 1 THEN amt ELSE 0 END), 2) as fraud_dollar_volume
    FROM transactions
    WHERE category = 'shopping_net'
    GROUP BY merchant
    ORDER BY fraud_dollar_volume DESC
    LIMIT 15
""").df()

,merchant,total_transactions,fraud_count,fraud_rate_pct,fraud_dollar_volume
0,fraud_Kozey-Boehm,1866,48.0,2.57,48189.98
1,fraud_Cormier LLC,1959,45.0,2.30,44845.95
2,fraud_Jast Ltd,1953,42.0,2.15,42560.34
3,fraud_Terry-Huel,1996,43.0,2.15,42356.37
4,fraud_Goyette Inc,1943,42.0,2.16,41580.84
5,fraud_Kerluke-Abshire,1838,41.0,2.23,40909.57
6,"fraud_Schmeler, Bashirian and Price",1968,41.0,2.08,40143.05
7,fraud_Gleason-Macejkovic,2033,40.0,1.97,39892.84
8,"fraud_Kuhic, Bins and Pfeffer",2003,39.0,1.95,39865.69
9,fraud_Kuhic LLC,1985,39.0,1.96,39765.06


In [29]:
# finding volume for category with the lowest percentage of fraud rate
con.execute("""
    SELECT 
        merchant,
        COUNT(*) as total_transactions,
        SUM(is_fraud) as fraud_count,
        ROUND(100.0 * SUM(is_fraud) / COUNT(*), 2) as fraud_rate_pct,
        ROUND(SUM(CASE WHEN is_fraud = 1 THEN amt ELSE 0 END), 2) as fraud_dollar_volume
    FROM transactions
    WHERE category = 'health_fitness'
    GROUP BY merchant
    ORDER BY fraud_dollar_volume DESC
    LIMIT 15
""").df()

,merchant,total_transactions,fraud_count,fraud_rate_pct,fraud_dollar_volume
0,fraud_Ratke and Sons,1723,7.0,0.41,143.92
1,"fraud_Runte, Green and Emard",1667,6.0,0.36,123.56
2,fraud_Ziemann-Waters,1695,6.0,0.35,112.29
3,fraud_Jacobi Inc,1665,5.0,0.30,102.12
4,"fraud_Ledner, Hartmann and Feest",1706,5.0,0.29,99.85
5,fraud_Bahringer Group,1722,5.0,0.29,99.40
6,fraud_Hirthe-Beier,1807,5.0,0.28,94.72
7,fraud_Reilly and Sons,1682,4.0,0.24,80.60
8,"fraud_Conroy, Balistreri and Gorczany",1739,4.0,0.23,80.35
9,"fraud_Dare, Fritsch and Zboncak",1740,4.0,0.23,77.55


In [30]:
# QUESTION 2: When is fraud occuring?
# HYPOTHESIS: Fraud will occur more during during the hours of 12-3am (late night or early morning). Stolen credit card information 
# might be used by fraudsters in a different time zone or are trying to avoid cardholders noticing these fradulent transactions since, 
# duringn these hours, the cardholders would be asleep.
# looking at when transactions occur fraud vs no_fraud
con.execute("""
    SELECT 
        trans_hour,
        SUM(CASE WHEN is_fraud = 0 THEN 1 ELSE 0 END) as legit_count,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) as fraud_count,
        ROUND(100.0 * SUM(is_fraud) / COUNT(*), 2) as fraud_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN is_fraud = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) as legit_rate_pct

    FROM transactions
    GROUP BY trans_hour
    ORDER BY trans_hour ASC
""").df()

,trans_hour,legit_count,fraud_count,fraud_rate_pct,legit_rate_pct
0,0,41867.0,635.0,1.49,98.51
1,1,42211.0,658.0,1.53,98.47
2,2,42031.0,625.0,1.47,98.53
3,3,42160.0,609.0,1.42,98.58
4,4,41817.0,46.0,0.11,99.89
5,5,42111.0,60.0,0.14,99.86
6,6,42260.0,40.0,0.09,99.91
7,7,42147.0,56.0,0.13,99.87
8,8,42456.0,49.0,0.12,99.88
9,9,42138.0,47.0,0.11,99.89


In [ ]:
# FINDINGS: The fraud rate increased significantly (2.88-0.11 = 2.77) from 9pm to 10pm. This supports the hypothesis that 
# fraud is occurring during late night hours. 

In [33]:
# QUESTION 3: What demographic group is more likely to be targeted by fraud?
# HYPOTHESIS: Older cardholders might be more likely to be targeted by fraud due to less knowledge about
# secure practices, leading to their information getting stolen. Higher income/credit limit might also contribute. 
con.execute("""
    SELECT 
        CASE 
            WHEN age < 25 THEN 'Under 25'
            WHEN age BETWEEN 25 AND 34 THEN '25-34'
            WHEN age BETWEEN 35 AND 44 THEN '35-44'
            WHEN age BETWEEN 45 AND 54 THEN '45-54'
            WHEN age BETWEEN 55 AND 64 THEN '55-64'
            ELSE '65+'
        END as age_group,
        COUNT(*) as total_transactions,
        SUM(is_fraud) as fraud_count,
        ROUND(100.0 * SUM(is_fraud) / COUNT(*), 2) as fraud_rate_pct
    FROM (
        SELECT *,
            DATE_DIFF('year', CAST(dob AS DATE), CURRENT_DATE) as age
        FROM transactions
    )
    GROUP BY age_group
    ORDER BY fraud_rate_pct DESC
""").df()

,age_group,total_transactions,fraud_count,fraud_rate_pct
0,65+,308354,2322.0,0.75
1,55-64,207810,1420.0,0.68
2,25-34,168073,1035.0,0.62
3,35-44,304161,1376.0,0.45
4,Under 25,13430,60.0,0.45
5,45-54,294847,1293.0,0.44


In [ ]:
# FINDINGS: Cardholders in age group 65+ are the traget demographic of fraud. 